# Stage 00 · Vulkan GPU Puzzles — Puzzle 01: Map (`@vnp.kernel`)

Welcome to **Vulkan GPU Puzzles**! Instead of only *calling* pre-built AI libraries, this track teaches you **how GPU kernels actually work** — in clean **Python syntax** on the Vulkan GPU you already own (no CUDA, ROCm, or Mojo required).

---

## How Does Python Code Run on a Vulkan GPU?

Your GPU does not run Python bytecode directly — Vulkan GPUs execute **SPIR-V** binary compute shaders. To let you write GPU kernels in Python, `vkdata` runs a 4-stage compiler pipeline inside [`vknumpy.kernel`](../../vknumpy/kernel.py):

| Stage | Component | Input → Output | What Happens |
|---|---|---|---|
| **1. Parse** | Python `ast.parse` | Python function → **AST** | Turns your `@vnp.kernel` function into a syntax tree (`FunctionDef`, `If`, `Subscript`, `BinOp`) |
| **2. Transpile** | `_GLSLTranspiler` | Python AST → **GLSL 450** | Maps arrays to Vulkan SSBOs (`binding=0,1,...`), typed scalars to Push Constants, `global_id_x()` to `gl_GlobalInvocationID.x`, and safely renames reserved GLSL words (`out` → `_out`) |
| **3. Compile** | `glslangValidator` | GLSL 450 → **SPIR-V** | Compiles GLSL into portable binary GPU bytecode (~20 ms, cached once per kernel) |
| **4. Dispatch** | `vksp_dispatch_spv` | SPIR-V + `vnp.ndarray`s → **GPU** | Caches the `VkPipeline` and runs **zero-copy** directly on your `vknumpy` `VkBuffer`s |


---
## Puzzle 01: `Map` — Add `10.0` to Every Element in Parallel

**Goal:** Given an input GPU array `a` of length `n`, compute `out[i] = a[i] + 10.0` for every index `0 <= i < n`.

**Key GPU Concepts:**
- **Workgroup (`local_size=(64, 1, 1)`)**: The GPU launches threads in teams of 64.
- **Grid (`(n + 63) // 64`)**: Number of workgroups needed to cover all `n` elements (rounded *up*, so the last workgroup may have extra threads past `n`!).
- **Thread ID (`i = global_id_x()`)**: Each thread gets its own unique global index `i`.
- **Bounds Guard (`if i < n:`)**: Extra threads in the final workgroup must do nothing so they don't write out of bounds!

In [1]:
import sys, time
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "vknumpy").is_dir()), Path.cwd())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import vknumpy as vnp
from vknumpy import global_id_x

# --- Puzzle 01 Kernel (written in pure Python syntax!) ---
@vnp.kernel(local_size=(64, 1, 1))
def puzzle_01_map(out, a, n: int):
    i = global_id_x()
    if i < n:
        out[i] = a[i] + 10.0

print(f"Compiled `{puzzle_01_map.name}` -> {len(puzzle_01_map.spirv_bytes):,} bytes of SPIR-V bytecode!")

Compiled `puzzle_01_map` -> 1,356 bytes of SPIR-V bytecode!


### Under the Hood: Inspect the Python AST & Generated GLSL 450

Let's look at the exact **Python AST** that `ast.parse` produced and the **GLSL 450 compute shader** our transpiler emitted from it!

In [2]:
print("=== 1. Python AST (first 24 lines) ===")
print("\n".join(puzzle_01_map.ast_dump.splitlines()[:24]))
print("   ...")
print("\n=== 2. Generated Vulkan GLSL 450 Shader ===")
print(puzzle_01_map.glsl)

=== 1. Python AST (first 24 lines) ===
FunctionDef(
  name='puzzle_01_map',
  args=arguments(
    posonlyargs=[],
    args=[
      arg(arg='out'),
      arg(arg='a'),
      arg(
        arg='n',
        annotation=Name(id='int', ctx=Load()))],
    kwonlyargs=[],
    kw_defaults=[],
    defaults=[]),
  body=[
    Assign(
      targets=[
        Name(id='i', ctx=Store())],
      value=Call(
        func=Name(id='global_id_x', ctx=Load()),
        args=[],
        keywords=[])),
    If(
      test=Compare(
        left=Name(id='i', ctx=Load()),
   ...

=== 2. Generated Vulkan GLSL 450 Shader ===
#version 450
layout(local_size_x = 64, local_size_y = 1, local_size_z = 1) in;

layout(set = 0, binding = 0) buffer Buf__out { float _out[]; };
layout(set = 0, binding = 1) buffer Buf_a { float a[]; };

layout(push_constant) uniform PC {
    int n;
} pc;

void main() {
    int i = int(gl_GlobalInvocationID.x);
    if (((i < pc.n))) {
        _out[i] = (a[i] + 10.0);
    }
}



### Run the Auto-Grader & Benchmark on 1,000,000 Elements

We test both a **non-multiple-of-64 size (`n = 1,000`)** to verify our bounds guard AND benchmark **1,000,000 elements** on your Vulkan GPU!

In [3]:
def check_puzzle_01(kernel_fn):
    for n in (1, 17, 64, 1000, 10_000):
        a_np = np.arange(n, dtype=np.float32)
        a = vnp.array(a_np)
        out = vnp.zeros((n,), dtype=np.float32)
        grid = (n + 63) // 64
        kernel_fn[grid](out, a, n)
        np.testing.assert_allclose(out.to_numpy(), a_np + 10.0, atol=1e-5)
    print("PASS! Puzzle 01 (Map) passed all 5 test sizes (1, 17, 64, 1000, 10000)!")

check_puzzle_01(puzzle_01_map)

# Benchmark on 1,000,000 elements
N = 1_000_000
a_big = vnp.array(np.arange(N, dtype=np.float32))
out_big = vnp.zeros((N,), dtype=np.float32)
grid_big = (N + 63) // 64

puzzle_01_map[grid_big](out_big, a_big, N)  # warmup (caches VkPipeline)
t0 = time.perf_counter()
for _ in range(50):
    puzzle_01_map[grid_big](out_big, a_big, N)
ms = (time.perf_counter() - t0) * 1000.0 / 50.0
gb_s = (2 * N * 4) / (ms * 1e-3) / 1e9
print(f"1,000,000 elements on Vulkan GPU: {ms:.2f} ms/launch (~{gb_s:.1f} GB/s effective bandwidth)")
print("First 6 outputs:", out_big.to_numpy()[:6])

PASS! Puzzle 01 (Map) passed all 5 test sizes (1, 17, 64, 1000, 10000)!
1,000,000 elements on Vulkan GPU: 0.24 ms/launch (~32.9 GB/s effective bandwidth)
First 6 outputs: [10. 11. 12. 13. 14. 15.]


ggml_vulkan: Found 1 Vulkan devices:
ggml_vulkan: 0 = AMD Custom GPU 0405 (RADV VANGOGH) (radv) | uma: 1 | rebar: 1 | fp16: 0 | bf16: 0 | fp4: 0 | warp size: 32 | shared memory: 65536 | int dot: 1 | matrix cores: none
